
# Dificultad de Ejercicios + Catálogo en Español — ASCEND

Reconstruye la dificultad (beginner/intermediate/expert) del dataset
principal de ejercicios (hasaneyldrm, con GIFs animados pero sin
dificultad), usando Free Exercise DB como fuente de verdad parcial +
un modelo supervisado + un LLM como segunda opinión. Al final traduce
todo al español y guarda el catálogo procesado.


1. **Panel de control al inicio** con dos interruptores:
   - `CORRER_SCORING_LLM`: en `False` por default — no vuelve a llamar
     a la API de OpenAI (esa clasificación ya se corrió previamente).
   - `SOBRESCRIBIR_PARQUET`: en `False` 

2. **Nueva sección**: descarga de los 1,324 GIFs animados a
   `Data/raw/ejercicios/` (Esto para fácilitar su descarga y depósito)


In [1]:

# ---------------------------------------------------------------------------
# IMPORTS
# ---------------------------------------------------------------------------
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests


/home/david-diaz/ds-venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


In [3]:

# ---------------------------------------------------------------------------
# VARIABLES DE ENTORNO — se cargan una sola vez aquí (antes se cargaban dos
# veces: aquí y otra vez en la sección del LLM). Ya no se imprime ningún
# fragmento de la key real: solo confirmamos que SÍ se cargó, sin mostrar
# su contenido en el output del notebook.
# ---------------------------------------------------------------------------
from dotenv import load_dotenv
import os

load_dotenv(".env")  # le decimos explícitamente el nombre real del archivo

_openai_key_detectada = bool(os.environ.get("OPENAI_API_KEY"))
print(f"OPENAI_API_KEY detectada: {'✅' if _openai_key_detectada else '❌ (revisa apis.env)'}")


OPENAI_API_KEY detectada: ✅


In [4]:

# ---------------------------------------------------------------------------
# PANEL DE CONTROL — los dos interruptores que decides tú antes de correr
# el resto del notebook. Todo lo demás respeta estos valores.
# ---------------------------------------------------------------------------

# False (default): NO vuelve a llamar a la API de OpenAI. Usa el checkpoint
# que ya tienes en disco (Data/processed/dificultad_llm_checkpoint.csv) tal
# cual está. Cámbialo a True solo cuando quieras (re)clasificar ejercicios
# nuevos o corregir algo.
CORRER_SCORING_LLM = False

# False (default): NO sobreescribe exercises_catalog_english_version.parquet
# ni exercises_catalog_spanish_version.parquet. El notebook sigue calculando
# todo en memoria (puedes revisar `ej_es` al final igual), simplemente no
# toca los archivos ya guardados. Cámbialo a True cuando sí quieras que se
# actualicen de verdad.
SOBRESCRIBIR_PARQUET = False

print(f"CORRER_SCORING_LLM = {CORRER_SCORING_LLM}")
print(f"SOBRESCRIBIR_PARQUET = {SOBRESCRIBIR_PARQUET}")


CORRER_SCORING_LLM = False
SOBRESCRIBIR_PARQUET = False


In [6]:

def descargar_si_no_existe(url: str, ruta_local: Path, timeout: int = 30) -> bool:
    '''
    Descarga un archivo binario desde `url` a `ruta_local`, pero SOLO si
    todavía no existe ahí — evita volver a pegarle a la fuente remota cada
    vez que reinicias el kernel y corres el notebook desde arriba.

    Antes este chequeo estaba duplicado (una vez por cada dataset que se
    descargaba); ahora es una sola función que se reutiliza en los 3
    lugares del notebook que bajan algo de internet (2 JSON + los GIFs).

    Parameters
    ----------
    url : str
        URL completa del archivo a descargar.
    ruta_local : Path
        Dónde guardarlo. La carpeta padre debe existir ya — créala antes
        de llamar a esta función (ej. `ruta_local.parent.mkdir(parents=True, exist_ok=True)`).
    timeout : int, default 30
        Segundos antes de dar por fallida la petición.

    Returns
    -------
    bool
        True si se descargó en ESTA llamada. False si ya existía (no se
        volvió a pedir) o si la descarga falló — el error se imprime,
        nunca se relanza, para no tronar un loop completo de cientos de
        archivos por uno solo que falle.
    '''
    if ruta_local.exists():
        return False
    try:
        respuesta = requests.get(url, timeout=timeout)
        respuesta.raise_for_status()
        ruta_local.write_bytes(respuesta.content)
        return True
    except requests.RequestException as e:
        print(f"  ⚠️ Falló la descarga de '{ruta_local.name}': {e}")
        return False


|
## 1. Descarga de Free Exercise DB

El dataset "de referencia" — el que SÍ trae dificultad real (`level`:
beginner/intermediate/expert), licencia Unlicense (dominio público, sin
restricciones). Lo usamos más adelante para "prestarle" dificultad a los
ejercicios del dataset principal cuyo nombre haga match.


Nota: este posteriormente no se usa ya que asignaba una dificultad erronea 
y sesgaba de manera sustancial a las recomendaciones de rutinas realizadas.

In [7]:

CARPETA_RAW = Path("../Data/raw")
CARPETA_RAW.mkdir(parents=True, exist_ok=True)  # crea la carpeta si no existe, no truena si ya existe

URL_DATASET = "https://raw.githubusercontent.com/yuhonas/free-exercise-db/main/dist/exercises.json"
RUTA_LOCAL = CARPETA_RAW / "free_exercise_db.json"

if descargar_si_no_existe(URL_DATASET, RUTA_LOCAL):
    print(f"Descargado: {RUTA_LOCAL.resolve()}")
else:
    print(f"Ya existía (o falló) — revisa el aviso arriba si hubo error: {RUTA_LOCAL.resolve()}")


Ya existía (o falló) — revisa el aviso arriba si hubo error: /home/david-diaz/ds-venv/Modulo_5/Data/raw/free_exercise_db.json



## 2. Carga de Free Exercise DB y primer vistazo


In [8]:

with open(RUTA_LOCAL, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)
print(f"Dataset: {df.shape[0]} filas, {df.shape[1]} columnas")
print("\nColumnas:", list(df.columns))
print("\nPrimer registro completo:")
print(df.iloc[0])
# Lo importante aquí: confirmar que sí trae la columna 'level' (dificultad
# real) y 'name' (para el cruce de nombres que viene más adelante).


Dataset: 873 filas, 11 columnas

Columnas: ['name', 'force', 'level', 'mechanic', 'equipment', 'primaryMuscles', 'secondaryMuscles', 'instructions', 'category', 'images', 'id']

Primer registro completo:
name                                                       3/4 Sit-Up
force                                                            pull
level                                                        beginner
mechanic                                                     compound
equipment                                                   body only
primaryMuscles                                           [abdominals]
secondaryMuscles                                                   []
instructions        [Lie down on the floor and secure your feet. Y...
category                                                     strength
images                           [3_4_Sit-Up/0.jpg, 3_4_Sit-Up/1.jpg]
id                                                         3_4_Sit-Up
Name: 0, dtype: object



## 3. Descarga del dataset principal (hasaneyldrm, con GIFs)

Dataset "de producción" — 1,324 ejercicios con GIFs animados, pero
SIN dificultad. Todo lo que sigue en este notebook es reconstruir esa
dificultad faltante usando Free Exercise DB como apoyo.


In [9]:

URL_HASANEYLDRM = "https://raw.githubusercontent.com/hasaneyldrm/exercises-dataset/main/data/exercises.json"
RUTA_HASANEYLDRM = CARPETA_RAW / "exercises_hasaneyldrm.json"

if descargar_si_no_existe(URL_HASANEYLDRM, RUTA_HASANEYLDRM, timeout=60):
    print(f"Descargado: {RUTA_HASANEYLDRM.resolve()}")
else:
    print(f"Ya existía (o falló): {RUTA_HASANEYLDRM.resolve()}")


Ya existía (o falló): /home/david-diaz/ds-venv/Modulo_5/Data/raw/exercises_hasaneyldrm.json



## 4. Carga y Wrangling del dataset principal


In [10]:

with open(RUTA_HASANEYLDRM, "r", encoding="utf-8") as f:
    data_ejercicios = json.load(f)

ej = pd.DataFrame(data_ejercicios)
print(f"Dataset: {ej.shape[0]} filas, {ej.shape[1]} columnas")

# category y body_part vienen idénticas en el 100% de los registros (ya lo
# confirmamos en el EDA) — es una columna redundante, nos quedamos con category.
ej = ej.drop(columns=["body_part"])

# muscle_group tiene una inconsistencia de nomenclatura real: "traps" y
# "trapezius" son el mismo músculo pero aparecen como categorías separadas.
# Si más adelante encuentras más casos así en el EDA, agrégalos a este diccionario.
ej["muscle_group"] = ej["muscle_group"].replace({"traps": "trapezius"})

print("category:", ej["category"].unique())
print("\nmuscle_group:", sorted(ej["muscle_group"].unique()))
print("\ntarget:", sorted(ej["target"].unique()))


Dataset: 1324 filas, 15 columnas
category: ['waist' 'upper legs' 'back' 'lower legs' 'chest' 'upper arms' 'cardio'
 'shoulders' 'lower arms' 'neck']

muscle_group: ['abdominals', 'ankle stabilizers', 'ankles', 'biceps', 'calves', 'chest', 'core', 'deltoids', 'forearms', 'glutes', 'hamstrings', 'hands', 'hip flexors', 'latissimus dorsi', 'lats', 'lower back', 'obliques', 'quadriceps', 'rhomboids', 'rotator cuff', 'shoulders', 'soleus', 'trapezius', 'triceps', 'upper back', 'wrist extensors', 'wrist flexors', 'wrists']

target: ['abductors', 'abs', 'adductors', 'biceps', 'calves', 'cardiovascular system', 'delts', 'forearms', 'glutes', 'hamstrings', 'lats', 'levator scapulae', 'pectorals', 'quads', 'serratus anterior', 'spine', 'traps', 'triceps', 'upper back']



## 4.5 Descarga de los GIFs animados

Van a `Data/raw/ejercicios/` (si no existe la carpeta se crea).

- Son 1,324 archivos


In [11]:

CARPETA_GIFS = CARPETA_RAW / "ejercicios"
CARPETA_GIFS.mkdir(parents=True, exist_ok=True)

# El repo expone /images (thumbnails .jpg) y /videos (animaciones .gif) en
# su raíz — el 'gif_url' del JSON ya trae "videos/archivo.gif", solo falta
# la base del repo.
BASE_URL_HASANEYLDRM_MEDIA = "https://raw.githubusercontent.com/hasaneyldrm/exercises-dataset/main/"

descargados, ya_existian, fallidos = 0, 0, []
total = len(ej)

for i, row in ej.iterrows():
    gif_relativo = row.get("gif_url")
    if not gif_relativo or (isinstance(gif_relativo, float) and pd.isna(gif_relativo)):
        continue  # por si algún registro no trae gif_url

    # Nos quedamos con el nombre de archivo original (ej. "0001-2gPfomN.gif")
    # para que coincida con la convención de nombres que ya usa 'image'.
    ruta_gif = CARPETA_GIFS / Path(gif_relativo).name

    if ruta_gif.exists():
        ya_existian += 1
        continue

    exito = descargar_si_no_existe(BASE_URL_HASANEYLDRM_MEDIA + gif_relativo, ruta_gif, timeout=20)
    if exito:
        descargados += 1
    else:
        fallidos.append(row["id"])

    if (i + 1) % 100 == 0:
        print(f"Progreso: {i + 1} de {total}")

    time.sleep(0.05)  # cortesía con el CDN — no es obligatorio, pero evita ráfagas agresivas

print(f"\nGIFs nuevos descargados: {descargados}")
print(f"Ya existían de antes: {ya_existian}")
print(f"Fallidos: {len(fallidos)}")
if fallidos:
    print(f"IDs que fallaron (reintenta corriendo esta celda de nuevo): {fallidos[:15]}{'...' if len(fallidos) > 15 else ''}")


Progreso: 100 de 1324
Progreso: 200 de 1324
Progreso: 300 de 1324
Progreso: 400 de 1324
Progreso: 500 de 1324
Progreso: 600 de 1324
Progreso: 700 de 1324
Progreso: 800 de 1324
Progreso: 900 de 1324
Progreso: 1000 de 1324
Progreso: 1100 de 1324
Progreso: 1200 de 1324
Progreso: 1300 de 1324

GIFs nuevos descargados: 1324
Ya existían de antes: 0
Fallidos: 0



## 5. Catalogación por zona muscular funcional

Esto NO es lo mismo que `category` — `category` es anatómica (pecho,
espalda, piernas...), pero para armar plantillas de rutina (empuje/
tracción/tren inferior/core) necesitas la clasificación FUNCIONAL, que
es la que se usa de verdad en programación de entrenamiento.

`"upper arms"` es el único caso ambiguo: agrupa tanto bíceps (músculo de
tracción, trabaja en jalones/remos) como tríceps (músculo de empuje,
trabaja en press). Por eso ese caso especial se resuelve mirando la
columna `target` en vez de `category`.


In [12]:

MAPEO_ZONA = {
    "chest": "Empuje superior",
    "shoulders": "Empuje superior",
    "back": "Tracción superior",
    "upper legs": "Tren inferior",
    "lower legs": "Tren inferior",
    "waist": "Core",
    "cardio": "Cardio",
    "lower arms": "Accesorio (antebrazo)",
    "neck": "Accesorio (cuello)",
}


def asignar_zona(row: pd.Series) -> str:
    '''
    Clasifica un ejercicio en su zona muscular FUNCIONAL (empuje/
    tracción/tren inferior/core/cardio/accesorio), a partir de sus
    columnas `category` y `target`.

    Caso especial: "upper arms" agrupa bíceps (tracción) y tríceps
    (empuje) bajo la misma `category`, así que para esos casos se
    desambigua mirando `target` en vez de `category`.

    Parameters
    ----------
    row : pd.Series
        Una fila del dataframe de ejercicios, debe traer "category" y "target".

    Returns
    -------
    str
        Una de las zonas de MAPEO_ZONA, o "Accesorio (brazo)" /
        "Sin clasificar" en los casos borde.
    '''
    if row["category"] == "upper arms":
        if row["target"] == "biceps":
            return "Tracción superior"
        if row["target"] == "triceps":
            return "Empuje superior"
        return "Accesorio (brazo)"  # otros músculos del brazo que no son bíceps/tríceps
    return MAPEO_ZONA.get(row["category"], "Sin clasificar")


ej["zona_muscular"] = ej.apply(asignar_zona, axis=1)

print("Distribución de zona_muscular:")
print(ej["zona_muscular"].value_counts())

# Sanity check: si algo cae en "Sin clasificar", significa que hay una
# 'category' que el diccionario MAPEO_ZONA no contempla — hay que revisarla.
print("\n¿Algo quedó 'Sin clasificar'? (debería ser 0)")
print(ej[ej["zona_muscular"] == "Sin clasificar"][["category", "target"]].value_counts())


Distribución de zona_muscular:
zona_muscular
Empuje superior          447
Tracción superior        354
Tren inferior            286
Core                     169
Accesorio (antebrazo)     37
Cardio                    29
Accesorio (cuello)         2
Name: count, dtype: int64

¿Algo quedó 'Sin clasificar'? (debería ser 0)
Series([], Name: count, dtype: int64)


Es importante esto ya que como vemos la mayor cantidad de ejercicios son del tren superior, lo que puede sesgarnos un poco las recomendaciones de rutinas por el volumen de ejercicios


## 6. Validación visual del mapeo — ¿`zona_muscular` tiene sentido vs `muscle_group`?

Tabla cruzada para confirmar a simple vista que, por ejemplo, "biceps"
cae en "Tracción superior" y no se mezcló con "Empuje superior" por error.


In [13]:

pd.crosstab(ej["zona_muscular"], ej["muscle_group"]).loc[
    :, (ej.groupby("muscle_group").size().sort_values(ascending=False).index)
]


muscle_group,shoulders,forearms,biceps,triceps,hamstrings,quadriceps,glutes,trapezius,obliques,hip flexors,...,abdominals,wrist flexors,latissimus dorsi,rhomboids,hands,ankle stabilizers,lats,upper back,wrist extensors,wrists
zona_muscular,,,,,,,,,,,,,,,,,,,,,
Accesorio (antebrazo),2,2,27,1,0,0,0,0,0,0,...,0,2,0,0,1,0,0,0,1,1
Accesorio (cuello),0,0,0,0,0,0,0,2,0,0,...,0,0,0,0,0,0,0,0,0,0
Cardio,1,0,0,0,0,25,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Core,24,2,0,0,0,4,5,0,63,65,...,0,0,0,0,0,0,1,0,0,0
Empuje superior,128,12,4,150,0,2,0,66,0,0,...,0,0,2,1,0,0,0,1,0,0
Tracción superior,36,149,133,10,4,2,13,0,1,0,...,2,0,0,1,0,0,0,0,0,0
Tren inferior,0,0,0,0,123,77,53,0,3,1,...,0,0,0,0,0,1,0,0,0,0



## 7. Match de nombres con Free Exercise DB — dificultad "prestada"

`rapidfuzz` compara qué tan parecidos son dos strings (0-100).
Normalizamos a minúsculas/sin espacios extra antes de comparar, porque
"3/4 Sit-Up" y "3/4 sit-up" deberían matchear aunque tengan mayúsculas
distintas.


In [14]:

from rapidfuzz import process, fuzz

df['name_norm'] = df['name'].str.lower().str.strip()
ej['name_norm'] = ej['name'].str.lower().str.strip()

choices = df['name_norm'].tolist()
nivel_por_nombre = dict(zip(df['name_norm'], df['level']))

# Umbral 90 ya lo validamos antes: por debajo de esto, el riesgo de heredar
# una dificultad de un ejercicio que en realidad es distinto sube mucho.
UMBRAL_MATCH_CONFIABLE = 90
dificultad_prestada = {}
for nombre in ej['name_norm']:
    match = process.extractOne(nombre, choices, scorer=fuzz.ratio)
    if match and match[1] >= UMBRAL_MATCH_CONFIABLE:
        dificultad_prestada[nombre] = nivel_por_nombre[match[0]]

ej['dificultad_prestada'] = ej['name_norm'].map(dificultad_prestada)
print(f"Match confiable: {ej['dificultad_prestada'].notna().sum()} de {len(ej)} "
      f"({ej['dificultad_prestada'].notna().mean()*100:.1f}%)")
# Esperado: ~13% — el resto (87%) necesita el modelo de la siguiente celda.


Match confiable: 177 de 1324 (13.4%)



## 8. Modelo supervisado de dificultad (2 clases) — entrenado con el match confiable

Por qué 2 clases y no 3: de los ~175 ejercicios con match confiable, solo
~10 son "expert" — insuficiente para que un modelo aprenda ese patrón de
forma confiable.


In [15]:

from sklearn.ensemble import RandomForestClassifier

ej["n_secondary"] = ej["secondary_muscles"].apply(len)  # cantidad de músculos secundarios, proxy de qué tan compuesto es el movimiento

etiquetados = ej[ej["dificultad_prestada"].notna()].copy()
etiquetados["dificultad_binaria"] = (etiquetados["dificultad_prestada"] != "beginner").astype(int)

# One-hot: el modelo no entiende texto, convierte cada categoría posible
# (cada equipo, cada zona, cada músculo) en su propia columna de 1s y 0s.
FEATURES_DIFICULTAD = ["category", "equipment", "muscle_group", "target"]
X_train_dif = pd.get_dummies(etiquetados[FEATURES_DIFICULTAD])
y_train_dif = etiquetados["dificultad_binaria"]

modelo_dificultad = RandomForestClassifier(
    n_estimators=300, max_depth=6, random_state=42, class_weight="balanced"
)
modelo_dificultad.fit(X_train_dif, y_train_dif)

print("Modelo entrenado con", len(etiquetados), "ejercicios etiquetados de forma confiable.")


Modelo entrenado con 177 ejercicios etiquetados de forma confiable.



## 9. Predicción para todos los ejercicios + sub-división final


In [16]:

X_todos = pd.get_dummies(ej[FEATURES_DIFICULTAD])
# reindex: el one-hot de "todos" puede tener categorías (equipos, músculos)
# que el de entrenamiento nunca vio, o viceversa. reindex alinea ambos
# conjuntos de columnas, rellenando con 0 lo que falte en cualquiera de los dos.
X_todos = X_todos.reindex(columns=X_train_dif.columns, fill_value=0)

ej["es_exigente_pred"] = modelo_dificultad.predict(X_todos)

# Sub-división de "más exigente" en intermedio/avanzado — esto YA NO es el
# modelo, es una regla simple de equipo. Menos confiable que la división
# principal (accesible vs. exigente), documentado así a propósito.
EQUIPO_AVANZADO = ["barbell", "ez barbell", "kettlebell", "smith machine"]


def nivel_final(row: pd.Series) -> str:
    '''
    Convierte la predicción binaria del modelo (0/1) en el nivel final
    de 3 categorías, usando una regla simple de equipo (no el modelo)
    para la sub-división intermedio/avanzado.

    Parameters
    ----------
    row : pd.Series
        Debe traer "es_exigente_pred" (0 o 1) y "equipment".

    Returns
    -------
    str
        "beginner" si el modelo dijo que no es exigente; si sí lo es,
        "expert" cuando el equipo está en EQUIPO_AVANZADO, si no
        "intermediate".
    '''
    if row["es_exigente_pred"] == 0:
        return "beginner"
    return "expert" if row["equipment"] in EQUIPO_AVANZADO else "intermediate"


ej["dificultad_final"] = ej.apply(nivel_final, axis=1)
# dificultad_fuente documenta de dónde salió cada etiqueta — importante para
# tu documento de metodología (transparencia sobre qué tan confiable es cada dato).
ej["dificultad_fuente"] = np.where(ej["dificultad_prestada"].notna(), "match_fdb", "modelo+regla")

print(ej["dificultad_final"].value_counts())
print("\nFuente:")
print(ej["dificultad_fuente"].value_counts(normalize=True).round(3))


dificultad_final
beginner        899
intermediate    299
expert          126
Name: count, dtype: int64

Fuente:
dificultad_fuente
modelo+regla    0.866
match_fdb       0.134
Name: proportion, dtype: float64



## 10. Revisión visual de una muestra


In [17]:

columnas_revision = [
    "name", "category", "equipment", "muscle_group", "target",
    "dificultad_prestada", "dificultad_final", "dificultad_fuente",
]
ej[columnas_revision].sample(15, random_state=1)


,name,category,equipment,muscle_group,target,dificultad_prestada,dificultad_final,dificultad_fuente
1160,sled 45° leg press (side pov),upper legs,sled machine,quadriceps,glutes,NaN,intermediate,modelo+regla
571,dumbbell lying pronation on floor,lower arms,dumbbell,biceps,forearms,NaN,intermediate,modelo+regla
65,band one arm standing low row,back,band,biceps,upper back,NaN,beginner,modelo+regla
115,barbell floor calf raise,lower legs,barbell,hamstrings,calves,NaN,beginner,modelo+regla
1253,swimmer kicks v. 2 (male),upper legs,body weight,hamstrings,glutes,NaN,intermediate,modelo+regla
446,cross body crunch,waist,body weight,obliques,abs,beginner,beginner,match_fdb
641,dumbbell rear fly,shoulders,dumbbell,trapezius,delts,NaN,intermediate,modelo+regla
852,incline twisting sit-up,waist,body weight,obliques,abs,NaN,beginner,modelo+regla
903,kettlebell seesaw press,shoulders,kettlebell,triceps,delts,intermediate,expert,match_fdb
857,inverted row bent knees,back,body weight,biceps,upper back,NaN,beginner,modelo+regla



## 11. Score de confianza del modelo — priorizar qué revisar a mano

`predict_proba` regresa, para cada ejercicio, qué % de los árboles del
bosque "votaron" por la clase 1 (más exigente). Cerca de 0 o de 1 = el
modelo está seguro; cerca de 0.5 = el modelo está indeciso, y son los
mejores candidatos para que revises tú a mano en vez de confiar
ciegamente.


In [18]:

ej["confianza_dificultad"] = modelo_dificultad.predict_proba(X_todos)[:, 1]

print("Predicciones con más confianza (el modelo está seguro):")
print(ej.reindex(
    (ej["confianza_dificultad"] - 0.5).abs().sort_values(ascending=False).index
)[columnas_revision + ["confianza_dificultad"]].head(10))

print("\nPredicciones más inciertas (revisar estas primero):")
print(ej.reindex(
    (ej["confianza_dificultad"] - 0.5).abs().sort_values().index
)[columnas_revision + ["confianza_dificultad"]].head(10))


Predicciones con más confianza (el modelo está seguro):
                                        name    category   equipment  \
907  kettlebell turkish get up (squat style)  upper legs  kettlebell   
899                  kettlebell pistol squat  upper legs  kettlebell   
887                   kettlebell front squat  upper legs  kettlebell   
888                  kettlebell goblet squat  upper legs  kettlebell   
890            kettlebell lunge pass through  upper legs  kettlebell   
906                      kettlebell thruster   shoulders  kettlebell   
382                          cable side bend       waist       cable   
383       cable side bend crunch (bosu ball)       waist       cable   
384                        cable side crunch       waist       cable   
414                    cable twist (up-down)       waist       cable   

    muscle_group  target dificultad_prestada dificultad_final  \
907   quadriceps  glutes        intermediate           expert   
899   quadriceps  glu


## 12. Clasificación vía LLM — segunda opinión sobre la dificultad

**Controlado por `CORRER_SCORING_LLM` (panel de control, arriba).**
Con `False`, esta sección y las dos siguientes (muestra de prueba +
loop completo) se SALTAN — no se llama a la API ni se gasta cuota/tiempo.
Las funciones se definen igual (no cuesta nada definirlas), pero nunca
se ejecutan si el flag está apagado.


In [19]:

MODELO_LLM = "gpt-5.4-mini"  # el modelo que sí tienes disponible en tu cuenta

RUBRICA_DIFICULTAD = '''
Asigna una puntuación de dificultad de 1 a 100 al siguiente ejercicio, considerando:
- Complejidad técnica y coordinación requerida.
- Riesgo de lesión si la ejecución es incorrecta.
- Fuerza/potencia necesaria para completarlo con buena forma.
- Si el ejercicio usa carga adicional (peso extra, lastre, chaleco con peso,
  banda de resistencia añadida) sobre una versión base del mismo movimiento,
  DEBE puntuar más alto que la versión sin esa carga adicional.

Guía de referencia (ancla, no exhaustiva):
- 1-15: ejercicios muy básicos, mínimo riesgo (ej. estiramientos, movilidad)
- 16-33: beginner — movimiento simple, mínima coordinación
- 34-50: intermediate bajo — técnica moderada, sin carga compleja
- 51-66: intermediate alto — coordinación y carga libre moderada
- 67-80: expert — movimiento compuesto complejo (ej. muscle-up sin peso extra)
- 81-100: expert avanzado — misma familia de movimiento pero con carga
  adicional o variante más demandante (ej. muscle-up con lastre)

Responde ÚNICAMENTE con un número entero del 1 al 100. Sin texto, sin
explicación, sin la palabra "score", solo el número.
'''


def score_a_tier(score: int) -> str:
    '''Convierte el score numérico (1-100) del LLM en una de las 3
    categorías finales, usando los mismos cortes que documenta la rúbrica
    (<=33 beginner, <=66 intermediate, resto expert).'''
    if score <= 33:
        return "beginner"
    elif score <= 66:
        return "intermediate"
    return "expert"


# El cliente solo se crea si de verdad vas a usarlo — así, si algún día
# corres este notebook sin apis.env configurado (ej. solo quieres ver el
# resto del pipeline), no truena aquí por una key faltante que no ibas a
# necesitar de todas formas.
client = None
if CORRER_SCORING_LLM:
    from openai import OpenAI
    client = OpenAI()  # busca OPENAI_API_KEY sola, ya cargada por load_dotenv


def clasificar_con_llm(nombre: str, equipo: str, instrucciones: str) -> tuple[int, str]:
    '''
    Le pide al LLM un score de dificultad (1-100) para un ejercicio, y lo
    convierte al tier de 3 categorías.

    Parameters
    ----------
    nombre : str
        Nombre del ejercicio.
    equipo : str
        Equipo requerido (ej. "barbell", "body weight").
    instrucciones : str
        Instrucciones en inglés del ejercicio (dan contexto de la
        ejecución real, no solo el nombre).

    Returns
    -------
    tuple[int, str]
        (score 1-100, tier "beginner"/"intermediate"/"expert").

    Raises
    ------
    ValueError
        Si el LLM responde algo que no sea un número entero puro — se
        deja tronar a propósito aquí, mejor detectarlo en el momento
        que meter basura silenciosamente al dataset.
    '''
    respuesta = client.chat.completions.create(
        model=MODELO_LLM,
        max_completion_tokens=10,
        temperature=0,
        messages=[
            {"role": "system", "content": RUBRICA_DIFICULTAD},
            {"role": "user", "content": f"Ejercicio: {nombre}\nEquipo: {equipo}\nInstrucciones: {instrucciones}"},
        ],
    )
    texto = respuesta.choices[0].message.content.strip()
    score = int(texto)
    return score, score_a_tier(score)



## 13. Prueba rápida sobre 5 ejercicios al azar

Se salta por completo si `CORRER_SCORING_LLM = False`.


In [20]:

if CORRER_SCORING_LLM:
    muestra = ej.sample(5, random_state=1)
    for _, row in muestra.iterrows():
        instrucciones_en = row["instructions"]["en"] if isinstance(row["instructions"], dict) else row.get("instructions_en", "")
        score, tier = clasificar_con_llm(row["name"], row["equipment"], instrucciones_en)
        print(f"{row['name']} ({row['equipment']}): score={score} -> tier='{tier}' | modelo estadístico decía '{row['dificultad_final']}'")
else:
    print("CORRER_SCORING_LLM está en False — se salta la prueba de 5 ejercicios. "
          "Cambia el flag en el panel de control si quieres correrla.")


CORRER_SCORING_LLM está en False — se salta la prueba de 5 ejercicios. Cambia el flag en el panel de control si quieres correrla.



## 14. Clasificación completa vía LLM (score + tier) — con checkpoint

También se salta por completo si `CORRER_SCORING_LLM = False` — en ese
caso, la siguiente sección (progreso) simplemente lee el checkpoint que
ya tengas guardado en disco, sin tocarlo.


In [21]:

Path("../Data/processed").mkdir(parents=True, exist_ok=True)
RUTA_CHECKPOINT = Path("../Data/processed/dificultad_llm_checkpoint.csv")

if not CORRER_SCORING_LLM:
    print("CORRER_SCORING_LLM está en False — no se llama a la API. "
          "Se va a usar el checkpoint existente tal cual (si lo hay) en la siguiente sección.")
else:
    # Si ya corriste esto antes y se cortó a medias, retoma desde ahí en vez de
    # empezar de cero (evita gastar de más si el kernel se cae a la mitad).
    if RUTA_CHECKPOINT.exists():
        resultados_llm = pd.read_csv(RUTA_CHECKPOINT)
        ids_ya_procesados = set(resultados_llm.loc[resultados_llm["score_llm"].notna(), "id"])
        print(f"Retomando checkpoint: {len(ids_ya_procesados)} ya procesados exitosamente.")
    else:
        resultados_llm = pd.DataFrame(columns=["id", "score_llm", "dificultad_llm", "respuesta_cruda"])
        ids_ya_procesados = set()

    filas_nuevas = []
    total = len(ej)

    for i, (_, row) in enumerate(ej.iterrows()):
        if row["id"] in ids_ya_procesados:
            continue

        instrucciones_en = row["instructions"]["en"] if isinstance(row["instructions"], dict) else row.get("instructions_en", "")

        try:
            score, tier = clasificar_con_llm(row["name"], row["equipment"], instrucciones_en)
            respuesta_cruda = None
        except Exception as e:
            # Guarda el texto crudo que sí llegó a devolver la API, si lo hubo,
            # para que puedas revisar después POR QUÉ falló (¿respondió texto en
            # vez de número? ¿truene de la API?) sin tener que adivinar.
            score, tier = None, None
            respuesta_cruda = str(e)
            print(f"Error en '{row['name']}': {e}")

        filas_nuevas.append({
            "id": row["id"], "score_llm": score, "dificultad_llm": tier,
            "respuesta_cruda": respuesta_cruda,
        })

        # Guarda cada 100 ejercicios, no solo al final — así si algo truena a la
        # mitad, no pierdes el trabajo ya pagado y hecho.
        if len(filas_nuevas) % 100 == 0:
            pd.concat([resultados_llm, pd.DataFrame(filas_nuevas)]).to_csv(RUTA_CHECKPOINT, index=False)
            print(f"Progreso: {i + 1} de {total}")

        time.sleep(0.3)  # evita saturar el rate limit de la API

    resultados_llm = pd.concat([resultados_llm, pd.DataFrame(filas_nuevas)]).drop_duplicates("id", keep="last")
    resultados_llm.to_csv(RUTA_CHECKPOINT, index=False)
    print(f"\nCompleto: {resultados_llm['score_llm'].notna().sum()} de {total} clasificados exitosamente.")


CORRER_SCORING_LLM está en False — no se llama a la API. Se va a usar el checkpoint existente tal cual (si lo hay) en la siguiente sección.



## 15. Progreso actual de la clasificación

Esta celda siempre corre igual, sin importar el flag — solo LEE el
checkpoint que exista en disco (no llama a ninguna API).


In [22]:

Path("../Data/processed").mkdir(parents=True, exist_ok=True)
RUTA_CHECKPOINT = Path("../Data/processed/dificultad_llm_checkpoint.csv")

if RUTA_CHECKPOINT.exists():
    checkpoint_actual = pd.read_csv(RUTA_CHECKPOINT)

    exitosos = checkpoint_actual["score_llm"].notna().sum()
    fallidos = checkpoint_actual["score_llm"].isna().sum()
    total = len(ej)
    procesados = len(checkpoint_actual)

    print(f"Total de ejercicios en el dataset: {total}")
    print(f"Procesados hasta ahora: {procesados} ({procesados/total*100:.1f}%)")
    print(f"  - Exitosos (con score): {exitosos}")
    print(f"  - Fallidos (sin score): {fallidos}")
    print(f"Faltan por procesar: {total - procesados}")

    if exitosos > 0:
        print("\nDistribución de dificultad (solo los exitosos hasta ahora):")
        print(checkpoint_actual["dificultad_llm"].value_counts())
        print("\nEstadísticos del score (1-100):")
        print(checkpoint_actual["score_llm"].describe().round(1))

    if fallidos > 0:
        print(f"\n{fallidos} ejercicios fallidos — revisa 'respuesta_cruda' para ver por qué:")
        print(checkpoint_actual[checkpoint_actual["score_llm"].isna()][["id", "respuesta_cruda"]].head(10))
else:
    print("Todavía no existe el checkpoint — activa CORRER_SCORING_LLM y corre la sección 14 primero.")


Total de ejercicios en el dataset: 1324
Procesados hasta ahora: 1324 (100.0%)
  - Exitosos (con score): 1324
  - Fallidos (sin score): 0
Faltan por procesar: 0

Distribución de dificultad (solo los exitosos hasta ahora):
dificultad_llm
intermediate    656
beginner        606
expert           62
Name: count, dtype: int64

Estadísticos del score (1-100):
count    1324.0
mean       36.2
std        15.4
min         5.0
25%        28.0
50%        38.0
75%        42.0
max        97.0
Name: score_llm, dtype: float64



## 16. Dataframe completo con la clasificación LLM

También corre siempre igual — usa lo que haya en el checkpoint (aunque
esté vacío o parcial), sin importar el flag.


In [23]:

if RUTA_CHECKPOINT.exists():
    checkpoint_actual = pd.read_csv(RUTA_CHECKPOINT)
else:
    checkpoint_actual = pd.DataFrame(columns=["id", "score_llm", "dificultad_llm"])

# El CSV le quitó los ceros a la izquierda a 'id' al guardarlo/leerlo
# (ej. "0001" se volvió 1). Forzamos ambos lados a string y rellenamos con
# ceros a la izquierda hasta 4 dígitos, que es el formato original de ej["id"].
checkpoint_actual["id"] = checkpoint_actual["id"].astype(str).str.zfill(4)
ej["id"] = ej["id"].astype(str).str.zfill(4)

ej_clasificado = ej.merge(
    checkpoint_actual[["id", "score_llm", "dificultad_llm"]] if len(checkpoint_actual) else pd.DataFrame(columns=["id", "score_llm", "dificultad_llm"]),
    on="id", how="left"
)

columnas_finales = [
    "id", "name", "category", "equipment", "muscle_group", "target",
    "dificultad_final", "score_llm", "dificultad_llm",
]

vista_ranking = ej_clasificado[columnas_finales].sort_values("score_llm", ascending=False)

print(f"Ejercicios clasificados vía LLM: {ej_clasificado['score_llm'].notna().sum()} de {len(ej_clasificado)}")
vista_ranking


Ejercicios clasificados vía LLM: 1324 de 1324


,id,name,category,equipment,muscle_group,target,dificultad_final,score_llm,dificultad_llm
812,3327,full planche push-up,chest,body weight,shoulders,pectorals,beginner,97,expert
810,3315,full maltese,waist,body weight,shoulders,abs,beginner,96,expert
1244,3314,straddle maltese,waist,body weight,shoulders,abs,beginner,95,expert
1294,3290,weighted one hand pull up,back,weighted,biceps,lats,intermediate,92,expert
1292,3286,weighted muscle up,back,weighted,biceps,lats,intermediate,92,expert
...,...,...,...,...,...,...,...,...,...
17,1709,assisted lying glutes stretch,upper legs,assisted,hamstrings,glutes,intermediate,8,beginner
1322,1428,wrist circles,lower arms,body weight,hands,forearms,beginner,8,beginner
1321,1604,world greatest stretch,upper legs,body weight,glutes,hamstrings,intermediate,8,beginner
6,1368,ankle circles,lower legs,body weight,ankle stabilizers,calves,beginner,8,beginner



## 17. Catálogo principal en español


In [24]:

TRADUCCION_CATEGORY = {
    "chest": "pecho", "back": "espalda", "shoulders": "hombros",
    "upper arms": "brazos (superior)", "lower arms": "antebrazo",
    "waist": "cintura/core", "upper legs": "piernas (superior)",
    "lower legs": "piernas (inferior)", "cardio": "cardio", "neck": "cuello",
}

TRADUCCION_EQUIPO = {
    "body weight": "peso corporal", "dumbbell": "mancuerna", "barbell": "barra",
    "cable": "polea", "kettlebell": "kettlebell", "leverage machine": "máquina de palanca",
    "ez barbell": "barra ez", "smith machine": "máquina smith", "band": "banda de resistencia",
    "assisted": "asistido", "medicine ball": "balón medicinal", "stability ball": "balón de estabilidad",
    "sled machine": "máquina de trineo", "olympic barbell": "barra olímpica",
    "resistance band": "banda de resistencia", "rope": "cuerda", "roller": "rodillo",
    # --- agregados por los warnings ---
    "upper body ergometer": "ergómetro de tren superior", "weighted": "con peso (lastrado)",
    "bosu ball": "balón bosu", "skierg machine": "máquina skierg",
    "hammer": "máquina hammer", "wheel roller": "rueda abdominal",
    "stationary bike": "bicicleta estática", "tire": "llanta",
    "trap bar": "barra hexagonal", "elliptical machine": "elíptica",
    "stepmill machine": "escaladora",
}

# Diccionario ÚNICO de músculos — cubre tanto los términos completos de
# muscle_group (ej. "quadriceps") como los abreviados de target (ej. "quads"),
# ya que ambas columnas usan el mismo diccionario (TRADUCCION_TARGET = TRADUCCION_MUSCLE_GROUP).
TRADUCCION_MUSCLE_GROUP = {
    "biceps": "bíceps", "triceps": "tríceps", "chest": "pecho", "abs": "abdominales",
    "quadriceps": "cuádriceps", "hamstrings": "isquiotibiales", "glutes": "glúteos",
    "calves": "pantorrillas", "lats": "dorsales", "trapezius": "trapecio",
    "forearm": "antebrazo", "deltoids": "deltoides", "adductors": "aductores",
    "abductors": "abductores", "neck": "cuello", "spine": "columna",
    "cardiovascular system": "sistema cardiovascular",
    # --- agregados por los warnings de muscle_group ---
    "hip flexors": "flexores de cadera", "obliques": "oblicuos",
    "ankle stabilizers": "estabilizadores de tobillo", "shoulders": "hombros",
    "forearms": "antebrazos", "lower back": "zona lumbar", "upper back": "espalda alta",
    "ankles": "tobillos", "core": "núcleo (core)", "rhomboids": "romboides",
    "rotator cuff": "manguito rotador", "wrist flexors": "flexores de muñeca",
    "wrist extensors": "extensores de muñeca", "latissimus dorsi": "dorsal ancho",
    "abdominals": "abdominales", "soleus": "sóleo", "wrists": "muñecas", "hands": "manos",
    # --- agregados por los warnings de target (términos abreviados) ---
    "quads": "cuádriceps", "pectorals": "pectorales", "delts": "deltoides",
    "traps": "trapecio", "serratus anterior": "serrato anterior",
    "levator scapulae": "elevador de la escápula",
}

TRADUCCION_TARGET = TRADUCCION_MUSCLE_GROUP  # mismo diccionario, ya cubre ambos vocabularios

TRADUCCION_DIFICULTAD = {
    "beginner": "principiante", "intermediate": "intermedio", "expert": "experto",
}


def traducir_columna(serie: pd.Series, diccionario: dict, nombre_columna: str) -> pd.Series:
    '''
    Traduce una columna completa usando un diccionario ES<-EN, avisando
    (sin tronar) qué valores no tenían traducción — esos se dejan tal
    cual en inglés en vez de perderlos o convertirlos en NaN.

    Parameters
    ----------
    serie : pd.Series
        La columna a traducir.
    diccionario : dict
        Mapeo inglés -> español.
    nombre_columna : str
        Solo para el mensaje de aviso (identificar qué columna fue).

    Returns
    -------
    pd.Series
        La columna traducida donde había mapeo; el valor original donde
        no lo había.
    '''
    traducida = serie.map(diccionario)
    faltantes = serie[traducida.isna()].unique()
    if len(faltantes) > 0:
        print(f"⚠️  '{nombre_columna}': {len(faltantes)} valores sin traducción, se dejaron en inglés: {list(faltantes)}")
    return traducida.fillna(serie)


ej_es = ej_clasificado.copy()
ej_es["category"] = traducir_columna(ej_es["category"], TRADUCCION_CATEGORY, "category")
ej_es["equipment"] = traducir_columna(ej_es["equipment"], TRADUCCION_EQUIPO, "equipment")
ej_es["muscle_group"] = traducir_columna(ej_es["muscle_group"], TRADUCCION_MUSCLE_GROUP, "muscle_group")
ej_es["target"] = traducir_columna(ej_es["target"], TRADUCCION_TARGET, "target")
ej_es["dificultad_final"] = traducir_columna(ej_es["dificultad_final"], TRADUCCION_DIFICULTAD, "dificultad_final")
ej_es["dificultad_llm"] = traducir_columna(ej_es["dificultad_llm"], TRADUCCION_DIFICULTAD, "dificultad_llm")

# ---------------------------------------------------------------------------
# GUARDAR AMBAS VERSIONES — solo si SOBRESCRIBIR_PARQUET = True
# ---------------------------------------------------------------------------
CARPETA_PROCESSED = RUTA_CHECKPOINT.parent
RUTA_CATALOGO_EN = CARPETA_PROCESSED / "exercises_catalog_english_version.parquet"
RUTA_CATALOGO_ES = CARPETA_PROCESSED / "exercises_catalog_spanish_version.parquet"

if SOBRESCRIBIR_PARQUET:
    ej_clasificado.to_parquet(RUTA_CATALOGO_EN, index=False)
    ej_es.to_parquet(RUTA_CATALOGO_ES, index=False)
    print(f"\nGuardado catálogo en inglés: {RUTA_CATALOGO_EN.resolve()}")
    print(f"Guardado catálogo en español: {RUTA_CATALOGO_ES.resolve()}")
else:
    print(f"\nSOBRESCRIBIR_PARQUET = False — NO se tocaron los archivos existentes:")
    print(f"  {RUTA_CATALOGO_EN.resolve()}")
    print(f"  {RUTA_CATALOGO_ES.resolve()}")
    print("Los dataframes 'ej_clasificado' y 'ej_es' sí están completos en memoria, por si quieres revisarlos abajo.")

print(f"\nFilas: {len(ej_es)} | Columnas: {list(ej_es.columns)}")



SOBRESCRIBIR_PARQUET = False — NO se tocaron los archivos existentes:
  /home/david-diaz/ds-venv/Modulo_5/Data/processed/exercises_catalog_english_version.parquet
  /home/david-diaz/ds-venv/Modulo_5/Data/processed/exercises_catalog_spanish_version.parquet
Los dataframes 'ej_clasificado' y 'ej_es' sí están completos en memoria, por si quieres revisarlos abajo.

Filas: 1324 | Columnas: ['id', 'name', 'category', 'equipment', 'instructions', 'instruction_steps', 'muscle_group', 'secondary_muscles', 'target', 'image', 'gif_url', 'media_id', 'created_at', 'attribution', 'zona_muscular', 'name_norm', 'dificultad_prestada', 'n_secondary', 'es_exigente_pred', 'dificultad_final', 'dificultad_fuente', 'confianza_dificultad', 'score_llm', 'dificultad_llm']
